In [22]:
import numpy as np
import pandas as pd

In [23]:
xtraindf = pd.read_csv('mnist_train.csv')
xtestdf = pd.read_csv('mnist_test.csv')
xtrain = xtraindf.to_numpy()
xtest =xtestdf.to_numpy() #first column is label

In [24]:
ytrains = xtrain[:, 0]      
xtrains = xtrain[:, 1:]     

ytests = xtest[:, 0]
xtests= xtest[:, 1:]

xtrain = xtrain / 255.0
xtest = xtest / 255.0

print(f"X train shape: {xtrains.shape}")
print(f"Y train shape: {ytrains.shape}")

X train shape: (59999, 784)
Y train shape: (59999,)


In [49]:
#z=wx+b
#

def relu(z):
    return np.maximum(0,z)

def reluder(z):
    return (z>0).astype(float)

def softmax(Z):
    return np.exp(Z)/np.sum(np.exp(Z), axis=0, keepdims=True)

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, 10))
    one_hot_Y[np.arange(Y.size), Y.astype(int)] = 1
    return one_hot_Y.T

X_trainfinal = xtrains.T 
Y_trainfinal = one_hot(ytrains)

def cost(expected,predicted):
    cost = -np.mean(expected * np.log(predicted + 1e-8))
    return cost

def initparams(layers): #layers is dimensions per layer
    weights = []
    biases = []
    for x in range(1,len(layers)):
        weights.append(np.random.randn(layers[x],layers[x-1])*0.01) #layer one at index 0 and so on
        biases.append(np.random.randn(layers[x],1))
    return [weights , biases]

def forwardpass(weights,biases,input):
    fi = [] #pre-relu
    hi = [] #post-relu
    hi.append(input)
    z=input
    for x in range(0,len(weights)-1):
        f = weights[x].dot(z) + biases[x]
        h = relu(f)
        fi.append(f)
        hi.append(h)
        z=h
    f = weights[-1].dot(z) + biases[-1]
    h = softmax(f)
    fi.append(f)
    hi.append(h)
    return fi,hi


def backprop(fi,hi,weights,biases,label,m):
    fiders = {}
    dz = hi[-1]-label
    fiders[len(fi)-1] = dz
    weightsgrads = [0]*len(weights)
    biasgrads = [0]*len(weights)
    for x in range(len(weights)-2,-1,-1):
        fiders[x] = (weights[x+1].T)@fiders[x+1]*reluder(fi[x])
    for x in range(len(fiders)):
        biasgrads[x] = np.sum(fiders[x], axis=1, keepdims=True)/m
        weightsgrads[x]= fiders[x]@(hi[x].T)/m
    return biasgrads,weightsgrads

def updateparams(weights,biases,learnrate,weightgrads,biasgrads):
    for x in range(len(weights)):
        weights[x] = weights[x] - weightgrads[x]*learnrate
        biases[x] = biases[x] - biasgrads[x]*learnrate
    return weights,biases

def r2(y_true, y_pred):
    y_mean = np.mean(y_true)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_mean) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    return r2
    



In [50]:
#training loop
def trainnn(input,layers,iters,output,learnrate):
    weights,biases = initparams(layers)
    for x in range(0,iters):
        fi,hi = forwardpass(weights,biases,input)
        biasgrad,weightgrad = backprop(fi,hi,weights,biases,output,input.size)
        prediction = hi[-1]
        losss = cost(output,prediction)
        updateparams(weights,biases,learnrate,weightgrad,biasgrad)
        if (x%100==0):
            print(f"epoch {x/100}:{losss}")
    
    return weights, biases

def predict(weights, biases, input_data):
    fi, hi = forwardpass(weights, biases, input_data)
    probabilities = hi[-1]
    predictions = np.argmax(probabilities, axis=0)
    return predictions
    

In [ ]:
layer_dims = [784, 128, 64, 10] 
weights, biases = trainnn(
    input=X_trainfinal, 
    layers=layer_dims, 
    iters=1000, 
    output=Y_trainfinal, 
    learnrate=0.1
)

train_predictions = predict(weights, biases, X_trainfinal)
accuracy = np.mean(train_predictions == ytrains) * 100

print(f"Training Accuracy: {accuracy:.2f}%")
print(r2(Y_trainfinal,train_predictions))

epoch 0.0:0.27944812008809944
